# BIST100 Piyasa Rejim Tespiti

Tikla ve hucreleri sirayla calistir.

In [ ]:
# Kurulum
import subprocess, sys
pkgs = ["yfinance","pandas","numpy","scikit-learn","hmmlearn","plotly"]
for p in pkgs:
    subprocess.run([sys.executable,"-m","pip","install","-q",p], check=True)
print("Tamamlandi!")

In [ ]:
# Kod
import warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, yfinance as yf
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from hmmlearn.hmm import GaussianHMM
import plotly.graph_objects as go

REGIME_LABELS = {0: "Risk-On", 1: "Carry Unwind", 2: "Stagflation Sideways"}

def fetch_data(tickers, period="5y"):
    data = {}
    for t in tickers:
        try:
            df = yf.Ticker(t).history(period=period)
            if not df.empty: data[t] = df
        except: pass
    return data

def create_features(data):
    primary = [t for t in ["XU100.IS","XBANK.IS","XUSIN.IS"] if t in data]
    if not primary: primary = list(data.keys())[0]
    else: primary = primary[0]
    df = data[primary].copy()
    df['returns'] = df['Close'].pct_change()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    df['volatility_20'] = df['returns'].rolling(20).std()
    df['volatility_63'] = df['returns'].rolling(63).std()
    df['momentum_20'] = df['Close']/df['Close'].shift(20)-1
    df['momentum_63'] = df['Close']/df['Close'].shift(63)-1
    delta = df['Close'].diff()
    gain = delta.where(delta>0,0).rolling(14).mean()
    loss = (-delta.where(delta<0,0)).rolling(14).mean()
    df['RSI'] = 100 - (100/(1+gain/loss))
    return df.dropna()

def train_kmeans(features, n=3):
    cols = [c for c in features.columns if c not in ['Open','High','Low','Close','Volume']]
    X = features[cols].values
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    km = KMeans(n_clusters=n, random_state=42, n_init=10)
    regimes = km.fit_predict(Xs)
    return km, scaler, cols, regimes

def train_hmm(features, cols, scaler):
    X = scaler.transform(features[cols].values)
    hmm = GaussianHMM(n_components=3, covariance_type="full", n_iter=100, random_state=42)
    hmm.fit(X)
    return hmm, hmm.predict(X)

def map_states(hmm_states, kmeans_regimes):
    state_map = {}
    for s in range(3):
        mask = hmm_states == s
        if mask.sum() > 0:
            most_common = pd.Series(kmeans_regimes[mask]).mode()[0]
            state_map[s] = REGIME_LABELS[most_common]
    return np.array([state_map[s] for s in hmm_states])

print("Fonksiyonlar hazir!")

In [ ]:
# Calistir
tickers = ["XU100.IS","XBANK.IS","XUSIN.IS","USDTRY=X","EEM"]
period = "5y"

print("Veri cekiliyor...")
data = fetch_data(tickers, period)
print(f"Toplanan veri: {len(data)} hisse")

print("Ozellikler hesaplaniyor...")
features = create_features(data)
print(f"Veri noktasi: {len(features)}")

print("KMeans egitiliyor...")
kmeans, scaler, cols, kmeans_regimes = train_kmeans(features)
print(f"Silhouette: {silhouette_score(scaler.transform(features[cols]), kmeans_regimes):.3f}")

print("HMM egitiliyor...")
hmm, hmm_states = train_hmm(features, cols, scaler)
hmm_regimes = map_states(hmm_states, kmeans_regimes)

# Current regime
X = scaler.transform(features[cols].iloc[-1:])
state = hmm.predict(X)[0]
state_map = {s: pd.Series(hmm_regimes[hmm_states==s]).mode()[0] for s in set(hmm_states)}
current_regime = state_map.get(state, "Unknown")
duration = sum(1 for s in reversed(hmm_states) if s == hmm_states[-1])

print("="*50)
print(f"GERECEL REJIM: {current_regime}")
print(f"Tarih: {features.index[-1].date()}")
print(f"Sure: {duration} gun")
print("="*50)

In [ ]:
# Grafikler
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=features.index, y=features['Close'], mode='lines', name='BIST100', line=dict(color='#00d4ff', width=2)))
fig1.update_layout(title='BIST100 Fiyat Grafiği', template='plotly_dark', height=400)
fig1.show()

# Performans
perf = {r: float(features.loc[hmm_regimes==r,'returns'].mean()*100) for r in set(hmm_regimes)}
fig2 = go.Figure(go.Bar(x=list(perf.keys()), y=list(perf.values()), marker_color=['#00d4ff','#ff6b6b','#ffd93d']))
fig2.update_layout(title='Rejim Performansi (%)', template='plotly_dark', height=350)
fig2.show()

# Geçis matrisi
fig3 = go.Figure(data=go.Heatmap(z=hmm.transmat_, colorscale='Blues', text=np.round(hmm.transmat_,2), texttemplate='%{text}', showscale=True))
fig3.update_layout(title='HMM Gecis Matrisi', template='plotly_dark', height=350)
fig3.show()

In [ ]:
# Dashboard HTML
from IPython.display import HTML
regimes = hmm_regimes
counts = pd.Series(regimes).value_counts()
total = len(regimes)
d = features.index[-1].date()

html = '''<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>BIST100 Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>body{font-family:Inter,sans-serif;background:#0d1117;color:#fff;margin:0;padding:20px}
.container{max-width:1000px;margin:0 auto}.header{text-align:center;padding:30px}
.header h1{background:linear-gradient(90deg,#00d4ff,#7b2cbf);-webkit-background-clip:text;-webkit-text-fill-color:transparent}
.card{background:#161b22;border-radius:15px;padding:25px;margin:20px 0;border:1px solid #30363d}
.grid{display:grid;grid-template-columns:repeat(4,1fr);gap:15px;margin:20px 0}
.stat{background:#21262d;padding:20px;border-radius:10px;text-align:center}
.stat-label{color:#8b949e;font-size:12px;text-transform:uppercase}
.stat-value{font-size:24px;font-weight:bold;color:#00d4ff}
</style></head>
<body><div class="container">
<div class="header"><h1>BIST100 Piyasa Rejim Dashboard</h1></div>
<div class="card"><h3>Guncel Piyasa Durumu</h3>
<div class="grid">
<div class="stat"><div class="stat-label">Rejim</div><div class="stat-value">'''+current_regime+'''</div></div>
<div class="stat"><div class="stat-label">Sure</div><div class="stat-value">'''+str(duration)+''' gun</div></div>
<div class="stat"><div class="stat-label">Tarih</div><div class="stat-value">'''+str(d)+'''</div></div>
<div class="stat"><div class="stat-label">Veri</div><div class="stat-value">'''+str(len(features))+'''</div></div>
</div></div>
<div class="card"><h3>Rejim Dagilimi</h3>
<div class="grid">
<div class="stat"><div class="stat-label">Risk-On</div><div class="stat-value">'''+f"{counts.get('Risk-On',0)/total*100:.1f}%"+'''</div></div>
<div class="stat"><div class="stat-label">Carry Unwind</div><div class="stat-value">'''+f"{counts.get('Carry Unwind',0)/total*100:.1f}%"+'''</div></div>
<div class="stat"><div class="stat-label">Stagflation</div><div class="stat-value">'''+f"{counts.get('Stagflation Sideways',0)/total*100:.1f}%"+'''</div></div>
</div></div></div></body></html>'''

display(HTML(html))